# 09 — Ecuación hacia atrás, mapeo conforme y validación Monte Carlo

**Repositorio de referencia:** `d2c4b91`.

El notebook anterior contenía tres exploraciones desconectadas:
transformaciones complejas, un ejemplo FEM 1D y un problema de Laplace con
un electrodo interno. Esta versión las reorganiza alrededor del objeto
matemático pertinente para narrow escape: la ecuación hacia atrás del
tiempo medio de primera salida.

Secciones:

1. transformación de Cayley para rectificar localmente la frontera;
2. problema de Poisson 1D con solución exacta, diferencias finitas y
   simulación browniana;
3. problema mixto 2D en el disco: Dirichlet en la ventana absorbente y
   Neumann en el resto de la frontera;
4. comparación del campo PDE con Monte Carlo mediante la API pública.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import sparse
from scipy.sparse.linalg import spsolve

from stage_escape import BrownianMotion, Escape, NaiveNarrowEscape, Surface

QUICK_RUN = True
N_JOBS = -1
BASE_SEED = 20260805
SAVE_FIGURES = False

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Ejecuta Jupyter dentro del repositorio STAGE.")

REPO_ROOT = find_repo_root()
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "notebook_09"
FIGURE_DIR = ARTIFACT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def finish_figure(fig, filename: str) -> None:
    fig.tight_layout()
    if SAVE_FIGURES:
        fig.savefig(FIGURE_DIR / f"{filename}.png", bbox_inches="tight")
        fig.savefig(FIGURE_DIR / f"{filename}.pdf", bbox_inches="tight")
    plt.show()

## 1. Rectificación conforme de la frontera

La transformación de Cayley

\[
w=i\frac{1+z}{1-z}
\]

envía el disco unitario al semiplano superior y la circunferencia a la
recta real, salvo el polo \(z=1\). Esta transformación no resuelve por sí
sola el problema estocástico, pero explica por qué una vecindad de una
ventana pequeña puede estudiarse en coordenadas localmente planas.

In [ ]:
def cayley(z, tolerance: float = 1.0e-12):
    z = np.asarray(z, dtype=np.complex128)
    denominator = 1.0 - z
    output = np.full(z.shape, np.nan + 1j * np.nan, dtype=np.complex128)
    np.divide(
        1j * (1.0 + z),
        denominator,
        out=output,
        where=np.abs(denominator) > tolerance,
    )
    return output.item() if output.ndim == 0 else output

theta = np.linspace(0.0, 2.0 * np.pi, 900, endpoint=False)
boundary = 0.999 * np.exp(1j * theta)

radial_curves = []
for angle in np.linspace(0.0, 2.0 * np.pi, 16, endpoint=False):
    radius = np.linspace(0.0, 0.98, 250)
    radial_curves.append(radius * np.exp(1j * angle))

circular_curves = []
for radius in np.linspace(0.15, 0.90, 6):
    circular_curves.append(radius * np.exp(1j * theta))

fig, ax = plt.subplots(figsize=(6.0, 6.0))
ax.plot(boundary.real, boundary.imag, linewidth=1.8)
for curve in radial_curves + circular_curves:
    ax.plot(curve.real, curve.imag, linewidth=0.6, alpha=0.7)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Re(z)")
ax.set_ylabel("Im(z)")
ax.set_title("Malla en el disco unitario")
finish_figure(fig, "cayley_domain")

fig, ax = plt.subplots(figsize=(8.0, 4.8))
for curve in radial_curves + circular_curves:
    image = cayley(curve)
    visible = np.isfinite(image) & (np.abs(image.real) <= 12.0) & (image.imag <= 12.0)
    ax.plot(image.real[visible], image.imag[visible], linewidth=0.7, alpha=0.75)
ax.axhline(0.0, linewidth=1.5)
ax.set_xlim(-12.0, 12.0)
ax.set_ylim(-0.25, 12.0)
ax.set_xlabel("Re(w)")
ax.set_ylabel("Im(w)")
ax.set_title("Imagen por Cayley: semiplano superior")
finish_figure(fig, "cayley_image")

## 2. Tiempo medio de salida en un intervalo

Para el generador \(D\partial_{xx}\), el tiempo medio de salida
\(u(x)=\mathbb E_x[\tau]\) del intervalo \((-L,L)\) satisface

\[
-D u''(x)=1,\qquad u(-L)=u(L)=0,
\]

cuya solución es

\[
u(x)=\frac{L^2-x^2}{2D}.
\]

In [ ]:
def interval_mfpt_exact(x, length: float = 1.0, diffusion: float = 1.0):
    x = np.asarray(x, dtype=float)
    return (length**2 - x**2) / (2.0 * diffusion)

def solve_interval_fd(
    n_intervals: int,
    length: float = 1.0,
    diffusion: float = 1.0,
):
    x = np.linspace(-length, length, n_intervals + 1)
    h = x[1] - x[0]
    n_internal = n_intervals - 1
    diagonal = np.full(n_internal, 2.0 * diffusion / h**2)
    off_diagonal = np.full(n_internal - 1, -diffusion / h**2)
    matrix = sparse.diags(
        [off_diagonal, diagonal, off_diagonal],
        offsets=[-1, 0, 1],
        format="csr",
    )
    rhs = np.ones(n_internal)
    solution = np.zeros_like(x)
    solution[1:-1] = spsolve(matrix, rhs)
    return x, solution

L = 1.0
D = 1.0
x_fd, u_fd = solve_interval_fd(80, length=L, diffusion=D)
u_exact = interval_mfpt_exact(x_fd, length=L, diffusion=D)

fig, ax = plt.subplots(figsize=(7.4, 4.6))
ax.plot(x_fd, u_fd, marker="o", markevery=5, label="diferencias finitas")
ax.plot(x_fd, u_exact, linestyle="--", label="solución exacta")
ax.set_xlabel("x")
ax.set_ylabel("E_x[tau]")
ax.set_title("Ecuación hacia atrás en un intervalo")
ax.legend()
finish_figure(fig, "interval_fd_exact")

convergence_rows = []
for n_intervals in [10, 20, 40, 80, 160, 320]:
    x_grid, numerical = solve_interval_fd(n_intervals, L, D)
    exact = interval_mfpt_exact(x_grid, L, D)
    convergence_rows.append(
        {
            "n_intervals": n_intervals,
            "h": 2.0 * L / n_intervals,
            "max_error": float(np.max(np.abs(numerical - exact))),
        }
    )
convergence = pd.DataFrame(convergence_rows)

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.loglog(convergence["h"], convergence["max_error"], marker="o")
ax.set_xlabel("h")
ax.set_ylabel("error máximo nodal")
ax.set_title("Convergencia de la discretización 1D")
finish_figure(fig, "interval_convergence")

convergence

In [ ]:
def interval_escape_sample(
    initial_x: float,
    delta_t: float,
    seed: int,
    max_steps: int,
    length: float = 1.0,
    diffusion: float = 1.0,
) -> float:
    surface = Surface(
        "interval",
        [lambda point: float(point[0] ** 2 - length**2)],
    )
    left = Escape([lambda point: bool(point[0] <= -length + 1.0e-10)])
    right = Escape([lambda point: bool(point[0] >= length - 1.0e-10)])
    motion = BrownianMotion(
        deposition_stride=100,
        delta_t=delta_t,
        dimension=1,
        D=diffusion,
        initial_position=np.array([initial_x]),
        seed=seed,
    )
    result = NaiveNarrowEscape(motion, surface, [left, right]).run(
        max_steps=max_steps
    )
    return np.nan if result.escape_time is None else result.escape_time

INITIAL_POINTS_1D = np.linspace(-0.8, 0.8, 9)
N_REPLICATES_1D = 40 if QUICK_RUN else 400
DELTA_T_1D = 5.0e-4
MAX_STEPS_1D = 1_000_000

seeds = np.random.SeedSequence(BASE_SEED).spawn(
    len(INITIAL_POINTS_1D) * N_REPLICATES_1D
)
tasks = []
cursor = 0
for initial_x in INITIAL_POINTS_1D:
    for _ in range(N_REPLICATES_1D):
        seed = int(seeds[cursor].generate_state(1)[0])
        cursor += 1
        tasks.append((float(initial_x), seed))

samples = Parallel(n_jobs=N_JOBS)(
    delayed(interval_escape_sample)(
        initial_x,
        DELTA_T_1D,
        seed,
        MAX_STEPS_1D,
        L,
        D,
    )
    for initial_x, seed in tasks
)

records = [
    {"initial_x": initial_x, "escape_time": value}
    for (initial_x, _), value in zip(tasks, samples, strict=True)
]
interval_mc = pd.DataFrame(records)
interval_summary = (
    interval_mc.groupby("initial_x")["escape_time"]
    .agg(["mean", "sem", "count"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7.4, 4.6))
fine_x = np.linspace(-L, L, 500)
ax.plot(
    fine_x,
    interval_mfpt_exact(fine_x, L, D),
    label="solución exacta/PDE",
)
ax.errorbar(
    interval_summary["initial_x"],
    interval_summary["mean"],
    yerr=1.96 * interval_summary["sem"],
    marker="o",
    linestyle="none",
    capsize=3,
    label="Monte Carlo, IC 95%",
)
ax.set_xlabel("posición inicial")
ax.set_ylabel("tiempo medio de salida")
ax.set_title("Validación cruzada PDE–Monte Carlo en 1D")
ax.legend()
finish_figure(fig, "interval_pde_monte_carlo")

interval_summary

## 3. Problema mixto 2D en el disco

Para una ventana absorbente \(\Gamma_a\) y el resto reflectante
\(\Gamma_r\), el tiempo medio satisface

\[
-D\Delta u=1\quad\text{en }\Omega,\qquad
u=0\quad\text{en }\Gamma_a,\qquad
\partial_nu=0\quad\text{en }\Gamma_r.
\]

Se usa una discretización cartesiana conservativa. Cuando un vecino cae
fuera del disco:

- en la ventana se aplica el valor de Dirichlet \(u=0\);
- en la frontera reflectante se usa un valor fantasma igual al nodo
  interior, que representa flujo normal nulo.

In [ ]:
def angle_distance(angle: float, center: float = 0.0) -> float:
    return float(np.abs(np.angle(np.exp(1j * (angle - center)))))

def solve_disk_mixed_fd(
    n: int = 101,
    radius: float = 1.0,
    diffusion: float = 1.0,
    window_half_angle: float = 0.16,
):
    coordinates = np.linspace(-radius, radius, n)
    h = coordinates[1] - coordinates[0]
    X, Y = np.meshgrid(coordinates, coordinates, indexing="xy")
    inside = X**2 + Y**2 <= (radius - 0.25 * h) ** 2

    index = -np.ones_like(X, dtype=int)
    inside_indices = np.argwhere(inside)
    index[inside] = np.arange(len(inside_indices))

    rows = []
    columns = []
    values = []
    rhs = np.full(len(inside_indices), h**2 / diffusion)

    neighbor_offsets = ((1, 0), (-1, 0), (0, 1), (0, -1))

    for row, (iy, ix) in enumerate(inside_indices):
        diagonal = 0.0
        x = X[iy, ix]
        y = Y[iy, ix]

        for dy, dx in neighbor_offsets:
            jy, jx = iy + dy, ix + dx
            neighbor_is_inside = (
                0 <= jy < n
                and 0 <= jx < n
                and inside[jy, jx]
            )

            if neighbor_is_inside:
                diagonal += 1.0
                rows.append(row)
                columns.append(index[jy, jx])
                values.append(-1.0)
                continue

            # Proyección radial del punto exterior para clasificar la frontera.
            qx = x + dx * h
            qy = y + dy * h
            boundary_angle = np.arctan2(qy, qx)
            absorbing = (
                angle_distance(boundary_angle, center=0.0)
                <= window_half_angle
            )

            if absorbing:
                # Contribución u_i - 0 de la condición de Dirichlet.
                diagonal += 1.0
            else:
                # Neumann homogénea: u_ghost = u_i, contribución nula.
                pass

        rows.append(row)
        columns.append(row)
        values.append(diagonal)

    matrix = sparse.csr_matrix(
        (values, (rows, columns)),
        shape=(len(inside_indices), len(inside_indices)),
    )
    solution_vector = spsolve(matrix, rhs)

    solution = np.full_like(X, np.nan, dtype=float)
    solution[inside] = solution_vector
    return coordinates, X, Y, inside, solution

WINDOW_HALF_ANGLE = 0.18
N_GRID = 81 if QUICK_RUN else 181

coordinates, X, Y, inside, U = solve_disk_mixed_fd(
    n=N_GRID,
    radius=1.0,
    diffusion=1.0,
    window_half_angle=WINDOW_HALF_ANGLE,
)

fig, ax = plt.subplots(figsize=(6.8, 6.0))
field = ax.pcolormesh(X, Y, np.ma.masked_invalid(U), shading="auto")
fig.colorbar(field, ax=ax, label="tiempo medio PDE")
theta = np.linspace(0.0, 2.0 * np.pi, 600)
ax.plot(np.cos(theta), np.sin(theta), linewidth=1.3)
absorbing_theta = np.linspace(-WINDOW_HALF_ANGLE, WINDOW_HALF_ANGLE, 100)
ax.plot(
    np.cos(absorbing_theta),
    np.sin(absorbing_theta),
    linewidth=4.0,
    label="ventana absorbente",
)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("MFPT por diferencias finitas con frontera mixta")
ax.legend()
finish_figure(fig, "disk_mixed_mfpt")

In [ ]:
midline_index = int(np.argmin(np.abs(coordinates)))
x_line = coordinates
u_line = U[midline_index, :]

fig, ax = plt.subplots(figsize=(7.4, 4.6))
ax.plot(x_line, u_line)
ax.set_xlabel("x")
ax.set_ylabel("u(x, 0)")
ax.set_title("Perfil del MFPT sobre el eje horizontal")
finish_figure(fig, "disk_midline_profile")

In [ ]:
def disk_escape_sample(
    initial_position: np.ndarray,
    seed: int,
    delta_t: float,
    max_steps: int,
    window_half_angle: float,
) -> float:
    surface = Surface(
        "unit disk",
        [lambda point: float(np.dot(point, point) - 1.0)],
    )
    window = Escape(
        [
            lambda point: bool(
                angle_distance(np.arctan2(point[1], point[0]), 0.0)
                <= window_half_angle + 1.0e-10
            )
        ]
    )
    motion = BrownianMotion(
        deposition_stride=100,
        delta_t=delta_t,
        dimension=2,
        D=1.0,
        initial_position=np.asarray(initial_position, dtype=float),
        seed=seed,
    )
    result = NaiveNarrowEscape(motion, surface, [window]).run(
        max_steps=max_steps
    )
    return np.nan if result.escape_time is None else result.escape_time

INITIAL_POINTS_2D = [
    np.array([-0.55, 0.0]),
    np.array([0.0, 0.0]),
    np.array([0.0, 0.55]),
]
N_REPLICATES_2D = 24 if QUICK_RUN else 250
DELTA_T_2D = 5.0e-4
MAX_STEPS_2D = 2_000_000

seeds = np.random.SeedSequence(BASE_SEED + 1).spawn(
    len(INITIAL_POINTS_2D) * N_REPLICATES_2D
)
tasks = []
cursor = 0
for point_id, point in enumerate(INITIAL_POINTS_2D):
    for _ in range(N_REPLICATES_2D):
        seed = int(seeds[cursor].generate_state(1)[0])
        cursor += 1
        tasks.append((point_id, point.copy(), seed))

escape_times = Parallel(n_jobs=N_JOBS)(
    delayed(disk_escape_sample)(
        point,
        seed,
        DELTA_T_2D,
        MAX_STEPS_2D,
        WINDOW_HALF_ANGLE,
    )
    for _, point, seed in tasks
)

comparison_rows = []
for point_id, point in enumerate(INITIAL_POINTS_2D):
    values = np.array(
        [
            value
            for (task_point_id, _, _), value in zip(tasks, escape_times, strict=True)
            if task_point_id == point_id
        ],
        dtype=float,
    )
    values = values[np.isfinite(values)]

    ix = int(np.argmin(np.abs(coordinates - point[0])))
    iy = int(np.argmin(np.abs(coordinates - point[1])))
    comparison_rows.append(
        {
            "point_id": point_id,
            "x": point[0],
            "y": point[1],
            "pde_value": U[iy, ix],
            "mc_mean": np.mean(values) if len(values) else np.nan,
            "mc_sem": (
                np.std(values, ddof=1) / np.sqrt(len(values))
                if len(values) > 1
                else np.nan
            ),
            "completed": len(values),
        }
    )

disk_comparison = pd.DataFrame(comparison_rows)

fig, ax = plt.subplots(figsize=(7.0, 4.8))
x_positions = np.arange(len(disk_comparison))
ax.plot(
    x_positions,
    disk_comparison["pde_value"],
    marker="o",
    label="PDE",
)
ax.errorbar(
    x_positions,
    disk_comparison["mc_mean"],
    yerr=1.96 * disk_comparison["mc_sem"],
    marker="s",
    linestyle="none",
    capsize=3,
    label="Monte Carlo, IC 95%",
)
ax.set_xticks(
    x_positions,
    [
        f"({row.x:.2f}, {row.y:.2f})"
        for row in disk_comparison.itertuples()
    ],
)
ax.set_ylabel("tiempo medio de escape")
ax.set_title("Comparación PDE–Monte Carlo en el disco")
ax.legend()
finish_figure(fig, "disk_pde_monte_carlo")

disk_comparison

## Límites numéricos

La discretización 2D es deliberadamente transparente y utiliza únicamente
NumPy/SciPy. No sustituye un análisis FEM de convergencia en una geometría
refinada cerca de los extremos de la ventana. Sus funciones son:

- comprobar la formulación de frontera mixta;
- producir una referencia independiente de Monte Carlo;
- revelar errores de signo, escala de \(D\), localización de la ventana o
  tratamiento reflectante.

Un estudio de alta precisión debería añadir refinamiento localizado,
estimación de error y comparación sistemática al variar `N_GRID`,
`DELTA_T_2D` y el tamaño angular de la ventana.